# Module 01 — Assignment: MLP + Backpropagation

This is hands-on, not a reading. You will implement the three moving parts of a
neural network **from scratch** — the forward pass, the backward pass (the chain
rule), and the gradient-descent step — and prove each one correct with an inline
check before moving on.

**What you'll build:** a `2 → 4 → 1` sigmoid MLP that finally solves XOR.

**How to work through it:**
1. Run the **Setup** cell once.
2. For each part: read the explanation, fill in the `# TODO` block (leave the
   `# GIVEN` lines alone), then run the **check** cell. Green ✓ = move on.
3. The **gradient check** in Part 2 is the heart of the assignment: it is how you
   *know* your hand-derived backprop is right, with no answer key needed.
4. Answer the **Inline Questions** in your own words.

> **On AI use:** deriving the gradients yourself is the entire point. Use the math
> in `notebook.ipynb` §2 as your reference, not an autocomplete.

In [ ]:
# Setup — run this once. (GIVEN: do not modify.)
import os, sys, math
import numpy as np

def _find(subdir, filename):
    here = os.getcwd()
    for c in [os.path.join(here, subdir),
              os.path.join(here, "topics", "01-mlp-backprop", subdir)]:
        if os.path.exists(os.path.join(c, filename)):
            return c
    raise FileNotFoundError(f"{filename} — run from the repo root or the module folder")

sys.path.insert(0, _find("tests", "check_utils.py"))
sys.path.insert(0, _find("python", "mlp.py"))
from check_utils import rel_error, eval_numerical_gradient, compare_to_canonical
import mlp as answer   # the answer key: python/mlp.py

# GIVEN: the deterministic RNG, the XOR data, and the parameter container. Do NOT
# reimplement Rng/make_xor — identical draws are what let you compare to the key.
Rng, make_xor = answer.Rng, answer.make_xor
X, y = make_xor()
XOR_SEED, N_HIDDEN, INIT_SCALE, XOR_LR, XOR_EPOCHS = 1, 4, 1.0, 0.5, 20000

class Params:
    # The learnable weights of a 2 -> H -> 1 net, drawn in a fixed order.
    def __init__(self, n_in, n_hidden, rng, scale=1.0):
        self.n_in, self.n_hidden = n_in, n_hidden
        self.W1 = np.zeros((n_hidden, n_in)); self.b1 = np.zeros(n_hidden)
        self.W2 = np.zeros(n_hidden);         self.b2 = np.zeros(1)
        for j in range(n_hidden):
            for k in range(n_in): self.W1[j, k] = scale * rng.signed()
        for j in range(n_hidden): self.b1[j] = scale * rng.signed()
        for j in range(n_hidden): self.W2[j] = scale * rng.signed()
        self.b2[0] = scale * rng.signed()

print("setup ok —", len(X), "XOR points, hidden width", N_HIDDEN)

## Part 1 — The forward pass

A hidden unit is just a Module-00 neuron: a weighted sum squashed by a sigmoid,
$\sigma(z) = 1/(1+e^{-z})$. Compute all `H` hidden activations, then combine them
into one sigmoid output.

$$a_{1,j} = \sigma\!\big(b_{1,j} + W_{1,j0}x_0 + W_{1,j1}x_1\big), \qquad
  \hat y = \sigma\!\Big(b_2 + \textstyle\sum_j W_{2,j}\,a_{1,j}\Big).$$

In [ ]:
def sigmoid(z):
    # GIVEN
    return 1.0 / (1.0 + math.exp(-z))

def forward(p, x0, x1):
    # Return (a1, a2): the hidden activations (a list) and the output.
    a1 = [0.0] * p.n_hidden
    #######################################################################
    # TODO: (1) for each hidden unit j, set a1[j] = sigmoid(z1) where      #
    #           z1 = b1[j] + W1[j,0]*x0 + W1[j,1]*x1                       #
    #       (2) set a2 = sigmoid(z2) where z2 = b2 + sum_j W2[j]*a1[j].    #
    #       Read weights as floats, e.g. float(p.W1[j, 0]).               #
    #######################################################################
    a2 = 0.0
    #######################################################################
    #                          END OF YOUR CODE                           #
    #######################################################################
    return a1, a2

# GIVEN: predict / loss / accuracy, all built on your forward().
def predict(p, x0, x1):
    _, a2 = forward(p, x0, x1); return 1 if a2 >= 0.5 else 0

def loss(p, X, y):
    n = len(y); total = 0.0
    for i in range(n):
        _, a2 = forward(p, float(X[i][0]), float(X[i][1])); yi = float(y[i])
        total += -(yi * math.log(a2) + (1.0 - yi) * math.log(1.0 - a2))
    return total / n

def accuracy(p, X, y):
    return sum(predict(p, float(X[i][0]), float(X[i][1])) == int(y[i])
               for i in range(len(y))) / len(y)

In [ ]:
# Check: your forward() must match the answer key on identical weights.
you = Params(2, N_HIDDEN, Rng(XOR_SEED), INIT_SCALE)
key = answer.MLP(2, N_HIDDEN, Rng(XOR_SEED), INIT_SCALE)
for p in X:
    _, ay = forward(you, *p)
    _, ky = key.forward(*p)
    print(f"x={p}  you={ay:.6f}  answer={ky:.6f}")
    assert rel_error(ay, ky) < 1e-12, "forward disagrees with the answer key"
print("forward matches the answer key ✓")

### Inline Question 1

An *untrained* network already produces outputs near 0.5 for every XOR point.
Why 0.5, and why does that mean it is guessing?

$\color{blue}{\textit{Your answer:}}$ *fill this in*

## Part 2 — Backpropagation by hand (the load-bearing part)

For binary cross-entropy through a sigmoid the output error is simply
$\delta_2 = a_2 - y$. Propagate it backward, using $\sigma'(z) = a(1-a)$:

$$\frac{\partial L}{\partial W_2} = \delta_2 a_1,\;\;
  \frac{\partial L}{\partial b_2} = \delta_2,\;\;
  \delta_1 = (W_2\,\delta_2)\odot a_1(1-a_1),\;\;
  \frac{\partial L}{\partial W_1} = \delta_1 x,\;\;
  \frac{\partial L}{\partial b_1} = \delta_1.$$

Accumulate over all four samples, then divide by `n` (the loss is a **mean**).

In [ ]:
def backward(p, X, y):
    # Return (dW1, db1, dW2, db2): the gradient of the mean loss.
    n = len(y)
    dW1 = np.zeros_like(p.W1); db1 = np.zeros_like(p.b1)
    dW2 = np.zeros_like(p.W2); db2 = np.zeros_like(p.b2)
    for i in range(n):
        x0, x1, yi = float(X[i][0]), float(X[i][1]), float(y[i])
        a1, a2 = forward(p, x0, x1)
        #######################################################################
        # TODO: accumulate this sample's contribution to every gradient.      #
        #   dz2 = a2 - yi                                                      #
        #   dW2[j] += dz2 * a1[j]        ;  db2[0] += dz2                       #
        #   da1 = dz2 * W2[j]                                                  #
        #   dz1 = da1 * a1[j] * (1 - a1[j])       # sigmoid'                   #
        #   dW1[j,0] += dz1*x0 ; dW1[j,1] += dz1*x1 ; db1[j] += dz1            #
        #######################################################################
        pass
        #######################################################################
        #                          END OF YOUR CODE                           #
        #######################################################################
    dW1 /= n; db1 /= n; dW2 /= n; db2 /= n
    return dW1, db1, dW2, db2

In [ ]:
# Check: compare your analytic gradient to a numerical one (finite differences).
# This needs NO answer key — the numerical gradient is ground truth.
p = Params(2, N_HIDDEN, Rng(XOR_SEED), INIT_SCALE)
dW1, db1, dW2, db2 = backward(p, X, y)
for name, P, analytic in [("W1", p.W1, dW1), ("b1", p.b1, db1),
                          ("W2", p.W2, dW2), ("b2", p.b2, db2)]:
    numeric = eval_numerical_gradient(lambda _: loss(p, X, y), P)
    err = rel_error(analytic, numeric)
    print(f"d{name:2s}  rel_error = {err:.2e}  {'✓' if err < 1e-6 else 'FAIL'}")
    assert err < 1e-6, f"{name} gradient is wrong"
print("backprop is correct ✓")

### Inline Question 2

If you forgot the $\sigma'(z) = a(1-a)$ factor when propagating into the hidden
layer, would the gradient check catch it? What roughly would `rel_error` look
like — near $10^{-7}$, or far larger — and why?

$\color{blue}{\textit{Your answer:}}$ *fill this in*

## Part 3 — Descend, and watch XOR fall

One gradient-descent step is $\theta \leftarrow \theta - \eta\,\partial L/\partial\theta$
for every parameter. Implement it, then the GIVEN loop trains for 20 000 steps.

In [ ]:
def sgd_step(p, X, y, lr):
    dW1, db1, dW2, db2 = backward(p, X, y)
    #######################################################################
    # TODO: subtract lr * gradient from each of p.W1, p.b1, p.W2, p.b2.    #
    #######################################################################
    pass
    #######################################################################
    #                          END OF YOUR CODE                           #
    #######################################################################

# GIVEN: the training loop.
p = Params(2, N_HIDDEN, Rng(XOR_SEED), INIT_SCALE)
for e in range(XOR_EPOCHS):
    sgd_step(p, X, y, XOR_LR)
print(f"final loss = {loss(p, X, y):.6f}   accuracy = {accuracy(p, X, y):.3f}")

### Inline Question 3

XOR needs a hidden layer, but *how many* units? Re-run Part 3 with `N_HIDDEN = 1`
(edit the Setup cell). Does it still reach accuracy 1.000? What does that tell you
about how many lines the hidden layer is drawing?

$\color{blue}{\textit{Your answer:}}$ *fill this in*

## Verify against the answer key

Finally, confirm your trained network matches the canonical `python/mlp.py` to
near double precision — same seed, same math, same numbers.

In [ ]:
wsum = float(p.W1.sum() + p.b1.sum() + p.W2.sum() + p.b2.sum())
you_fp = (loss(p, X, y), accuracy(p, X, y), wsum)
key_fp = dict(answer.run_xor())
compare_to_canonical(you_fp, (key_fp["loss"], key_fp["acc"], key_fp["wsum"]),
                     labels=["loss", "acc", "wsum"])